# 07 — Evaluation (Kaggle eval kernel)

Run the held-out competition benchmark, leakage checks, and base-capability regression probes on a Kaggle GPU, then write `output/evaluation-results.json` — the file the promotion gate (`scripts/verify_release.py`, via `scripts/evaluate_and_promote.py`) reads on the GitHub runner.

Mode is baked into this notebook at package time by `scripts/package_eval_kernel.py` (`eval-mode.json` is also written, but only this notebook file is uploaded to Kaggle, so nothing in-kernel may rely on it): `candidate` evaluates the trained adapter mounted from the training kernel (`/kaggle/input/`, resolved by searching for `adapter_config.json`); `base` evaluates the pinned base model, which is how the baseline gets measured (then record it with `scripts/record_baseline.py`). Scores are measured, never invented: `src/evaluation/solver.py` downloads each competition's train file with the Kaggle CLI, splits it into a fixed local train/val split (seed 42; chronological for time series), prompts the model once per competition for a solution, runs the solution sandboxed, and scores its val predictions against a deterministic reference computed on the same split. A competition whose data cannot be downloaded, whose solution times out, or whose submission is invalid is recorded unmeasured with null scores, the gate fails closed, and the candidate stays private. The leaderboard is never submitted to. The first code cell self-bootstraps: it clones this public repo and runs from the clone, so `src/`, `scripts/`, `configs/`, `evaluation/` and `data/` resolve in-kernel.

This notebook never publishes. Publication happens only on the GitHub runner, after the gate passes.

In [ ]:
# AKARUI_KERNEL_SELF_BOOTSTRAP
# Kaggle kernels receive ONLY this notebook file: `kaggle kernels push`
# uploads the single code_file from kernel-metadata.json, so the src/,
# scripts/, configs/, evaluation/ and data/ files the packager copies
# into kernels/<name>/ never reach the kernel. Self-bootstrap from the
# public repo instead (internet is enabled on the kernel) and run from
# the clone, so the relative paths below resolve. Idempotent: a re-run,
# or a local run from a repo root, skips the clone.
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Monibee-Fudgekins/Qwen3.8-27B-akarui"
REPO_DIR = Path("Qwen3.8-27B-akarui")
if (Path("src") / "__init__.py").is_file() and Path("scripts").is_dir():
    print("repo files already present; skipping clone")
elif (REPO_DIR / "src" / "__init__.py").is_file():
    os.chdir(REPO_DIR)
    print("repo root:", Path.cwd())
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print(f"cloning {REPO_URL} ...")
    try:
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True,
        )
    except OSError as exc:
        raise RuntimeError(f"repo clone failed (git not runnable): {exc}") from exc
    if result.returncode != 0:
        raise RuntimeError(
            "repo clone failed; the kernel cannot run without it: "
            + (result.stderr.strip() or result.stdout.strip())
        )
    os.chdir(REPO_DIR)
    print("cloned repo at:", Path.cwd())


In [ ]:
import sys
sys.path.insert(0, ".")
from pathlib import Path
print("repo root:", Path.cwd())
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_GPU = False
print("GPU available:", HAS_GPU)
if not HAS_GPU:
    print("Skipping GPU steps on this host — run this notebook on Kaggle with a GPU.")

In [ ]:
# AKARUI_KERNEL_INSTALL_REQUIREMENTS
# Install the kernel's declared dependencies (requirements-kaggle.txt).
# The Kaggle image ships torch/transformers but NOT bitsandbytes, so a
# 4-bit model load fails with ImportError unless we install it here
# (run #8, 2026-11-21). pip skips packages already satisfied, so this
# is fast when the image already has them. Skipped without a GPU so
# local/no-GPU runs stay light.
import subprocess, sys
if HAS_GPU:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"pip install requirements-kaggle.txt failed (exit {result.returncode})")
    print("requirements-kaggle.txt installed")
else:
    print("No GPU: dependency install skipped.")


In [ ]:
import json, os, subprocess, sys
from pathlib import Path

def _find_adapter_dir() -> Path | None:
    """Locate the trained adapter: Kaggle kernel input first."""
    env_dir = os.environ.get("ADAPTER_DIR")
    if env_dir and Path(env_dir).exists():
        return Path(env_dir)
    for candidate in Path("/kaggle/input").glob("**/adapter_config.json"):
        return candidate.parent
    local = Path("output/akarui-sft-v0")
    return local if local.exists() else None

mode = "candidate"
mode_file = Path("eval-mode.json")
if mode_file.exists():
    mode = json.loads(mode_file.read_text()).get("mode", mode)
mode = os.environ.get("EVAL_MODE", mode)
print("eval mode:", mode)

if HAS_GPU:
    if mode == "base":
        model_arg = "base"
    else:
        adapter_dir = _find_adapter_dir()
        if adapter_dir is None:
            raise RuntimeError(
                "candidate mode but no adapter found: expected the "
                "training kernel output under /kaggle/input/, "
                "$ADAPTER_DIR, or output/akarui-sft-v0"
            )
        model_arg = str(adapter_dir)
    print("evaluating model:", model_arg)
    result = subprocess.run(
        [sys.executable, "scripts/evaluate.py",
         "--config", "configs/evaluation.yaml",
         "--model", model_arg,
         "--output", "output/evaluation-results.json"],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"evaluation failed (exit {result.returncode})")
    results = json.loads(Path("output/evaluation-results.json").read_text())
    print("model:", results["model"])
    print("normalized score:", results["normalized_score"])
    print("submission validity:", results["submission_validity"])
    for entry in results["results"]:
        print("-", entry["competition_id"], entry["status"])
else:
    print("No GPU: evaluation skipped.")

In [ ]:
from src.evaluation.leakage_detection import detect_leakage
import json
manifest = json.load(open("evaluation/benchmark_manifest.json"))
held_out = [c["id"] for c in manifest["held_out_competitions"]]
print("held-out competitions:", len(held_out))
train_file = Path("data/train.jsonl")
if train_file.exists():
    train_ids = [json.loads(line).get("competition_id", "") for line in train_file.read_text().splitlines() if line.strip()]
    report = detect_leakage(train_ids, held_out)
    print("leakage report:", report)
    assert report["clean"], "leakage: held-out competition in training data"
else:
    print("No training file in this kernel; leakage was checked at packaging time.")


In [ ]:
import json
from pathlib import Path

results_path = Path("output/evaluation-results.json")
if results_path.exists():
    results = json.loads(results_path.read_text())
    regression = results.get("regression", {})
    print("regression probes:", regression.get("passed", 0), "passed /",
          regression.get("failed", 0), "failed /",
          regression.get("not_run", 0), "not run, of",
          regression.get("total", 0))
    print("per-category pass fractions:", results.get("regression_probes", {}))
    print("regressions vs baseline:", regression.get("regressions", []))
else:
    print("No evaluation results yet; the regression probes run inside "
          "scripts/evaluate.py with the same generation callable.")
